# Sampling the posterior

Erik, 2026-11-18

This notebook traces a single Stan regression from input preparation through
posterior sampling, first on simulated students and then on HSLS:09. The
HSLS:09 draws are saved at the end for the subsequent notebooks on posterior
summaries and posterior predictive checks.

Stan approximates the posterior distribution by Markov chain Monte Carlo: it
runs several independent sequences of iterations, called chains, whose recorded
draws form a sample from the posterior. Specifically, Stan uses Hamiltonian
Monte Carlo (HMC) with the No-U-Turn Sampler (NUTS). HMC treats the log
posterior density as a surface and simulates a particle moving across it along
its slope, allowing successive draws to land far apart in parameter space while
staying in regions of high posterior density. NUTS chooses the length of each
simulated path automatically, stopping once a trajectory begins to double back.
Each chain starts at a random point and begins with warmup, an initial tuning
phase that moves the sampler into the high-density region and calibrates both
its step size (the discrete increment advancing each trajectory) and a relative
scaling of the parameters. Once warmup ends, the chain records posterior draws.

In [ ]:
import json
import time
from pathlib import Path

import framework as fw
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

fw.scaffold()
import bayes  # functions/bayes.py, on the path after scaffold()

sns.set_theme(style="whitegrid", palette="colorblind")

clean = fw.data_read("inputs.private.final.hsls_clean", dtype={"student_id": str})
hsls = bayes.analysis_sample(clean)
regression = bayes.model("regression")

# One generator for the simulated students
rng = np.random.default_rng(505)

## The Stan program

The regression specification lives in `stan/regression.stan`, which is shared
with the R project. A Stan program divides a model into named blocks: `data`
declares the known inputs, `parameters` declares the unknown parameters to
estimate, `model` defines the log posterior density, and
`generated quantities` computes derived values from each draw.

In [ ]:
print((Path(fw.project_root()) / "stan" / "regression.stan").read_text())

## Data in

Stan expects its inputs in a dictionary whose keys match the declarations in
the `data` block, assembled from a data frame by `bayes.stan_data()`. For the
1,000 simulated students below, the output lists each entry's data type and
dimensions while omitting the underlying values.

In [ ]:
truth = {"alpha": 0.04, "beta": [0.7, 0.18], "sigma": 0.66}
fake = bayes.simulate(1000, truth["alpha"], truth["beta"], truth["sigma"], rng)
fake_data = bayes.stan_data(fake, bayes.WEAK)


def describe(data):
    rows = {}
    for key, value in data.items():
        array = np.asarray(value)
        rows[key] = {
            "Type": "integer" if np.issubdtype(array.dtype, np.integer) else "real",
            "Shape": "scalar" if array.ndim == 0 else " x ".join(str(d) for d in array.shape),
            "Value": value if array.ndim == 0 else (list(np.round(array, 2)) if array.size <= 2 else "(array)"),
        }
    return pd.DataFrame(rows).T


describe(fake_data)

In that summary, the predictor matrix `X` has one row per student and two
columns (base-year math in SD units, and SES as supplied), while `y` holds the
follow-up scores in SD units. The remaining entries are model switches and prior
settings documented in the Stan program's comments. Before sampling starts, Stan
checks the dictionary against the `data` block and halts the fit immediately if
an entry has the wrong shape or violates a declared bound, such as a negative
sample size `N`.

## Sampling

`bayes.sample()` runs four chains in parallel, each executing 1,000 warmup
iterations followed by 1,000 sampling iterations. Warmup draws are ordinarily
discarded, but this call retains them to show how the sampler adapts.

In [ ]:
start = time.perf_counter()
fake_fit = bayes.sample(regression, fake_data, save_warmup=True)
print(f"Sampling took {time.perf_counter() - start:.1f} seconds")
print(f"Output files: {len(fake_fit.runset.csv_files)} CSV files, one per chain")

## Draws out

Stan writes one CSV file per chain, with one row for each iteration. Every file
begins with sampler diagnostic columns ending in two underscores, listed in the
table below, followed by the model parameters and generated quantities. Those
leading columns record how each simulated trajectory behaved, including the log
posterior density (up to the normalizing constant omitted in Bayes' rule) and
numerical warnings, while the remaining columns hold the sampled quantities.

| Column | Meaning |
|---|---|
| `lp__` | Log posterior density at the draw, up to a constant |
| `accept_stat__` | Average acceptance probability of the trajectory |
| `stepsize__` | Step size of the trajectory, fixed after warmup |
| `treedepth__` | Depth of the trajectory tree; 2 to the depth bounds the number of steps |
| `n_leapfrog__` | Number of steps in the trajectory |
| `divergent__` | 1 when the trajectory diverged, a sign of a region the sampler cannot explore accurately |
| `energy__` | Hamiltonian energy, used by the E-BFMI diagnostic |

In [ ]:
draws = fake_fit.draws_pd(inc_warmup=False)
print(f"{len(draws):,} rows: 4 chains of 1,000 draws")
draws.head(5).round(3)

In [ ]:
# The same draws as an array: iterations by chains by columns
array = fake_fit.draws(inc_warmup=False)
print(f"Array shape: {array.shape}")
print(f"Columns: {fake_fit.column_names}")

## Warmup

The two-panel plot below follows adaptation across all 2,000 iterations of each
chain, with a vertical dashed line marking the end of warmup. The left panel
traces the SES slope: each chain leaves its random start early in warmup,
enters the region of high posterior density, and mixes within a common band
across the post-warmup iterations. The right panel traces the step size, which
adjusts during warmup while Stan tunes the sampler and stays fixed once
sampling begins.

In [ ]:
with_warmup = fake_fit.draws_pd(inc_warmup=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for chain, chain_draws in with_warmup.groupby("chain__"):
    axes[0].plot(chain_draws["iter__"], chain_draws["beta[2]"], linewidth=0.5, label=f"Chain {chain:.0f}")
    axes[1].plot(chain_draws["iter__"], chain_draws["stepsize__"], linewidth=1)
for ax in axes:
    ax.axvline(1000, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Iteration")
axes[0].axhline(truth["beta"][1], color="black", linewidth=1)
axes[0].set_ylabel("SES slope")
axes[0].set_title("SES slope, warmup and sampling (solid line: true value)")
axes[0].legend(fontsize=8, loc="upper right")
axes[1].set_yscale("log")
axes[1].set_ylabel("Step size (log scale)")
axes[1].set_title("Step size")
fig.tight_layout()

In [ ]:
# Early warmup iterations of each chain, on the way to the posterior
early = with_warmup[with_warmup["iter__"].isin([1, 5, 10, 25, 50, 100, 150, 200, 300])]
early[["chain__", "iter__", "lp__", "alpha", "beta[1]", "beta[2]", "sigma"]].round(2)

To sample without boundary constraints, Stan maps bounded parameters onto an
unconstrained scale spanning all real numbers, representing a positive standard
deviation by its logarithm. Each chain starts at random values between −2 and 2
on that unconstrained scale. In SD units, the posterior lies inside that range,
but a random start still lands well away from it: chain 3 starts with a
base-year math slope of −1.3, where the posterior is near 0.7. Far from the
posterior, proposals are often rejected, leaving a chain at its starting value
for several iterations before it begins to move; chain 3 is still at its start
at iteration 5. The table above tracks the chains across warmup as they climb
toward the posterior: `lp__` rises from the thousands below zero to about −100
while the parameters move into the range of the final draws, reached in this run
by iteration 25 in every chain.

Entering that high-density region early in warmup still falls short of
convergence, the condition that separate chains have mixed into the same
posterior distribution. The diagnostics in `06-diagnostics` check whether the
chains agree with one another after warmup. Warmup runs well past the initial
climb because tuning the step size and parameter scaling takes longer than
finding the posterior, and none of the warmup draws are used for inference.

## HSLS:09

The following cells run the same workflow on the 18,623 HSLS:09 students under
the weak priors. After sampling, `fw.save_model()` writes the parameter draws
to the private models folder and records the file in the project database so
`07-posterior-summary` and `08-posterior-predictive` can load them with
`fw.load_model()`.

In [ ]:
hsls_data = bayes.stan_data(hsls, bayes.WEAK)
display(describe(hsls_data))

start = time.perf_counter()
hsls_fit = bayes.sample(regression, hsls_data)
print(f"Sampling took {time.perf_counter() - start:.1f} seconds")

In [ ]:
hsls_draws = bayes.draws(hsls_fit)
saved = fw.save_model(hsls_draws, "hsls_regression_draws")

# What else a reader of the draws needs: the priors and how the variables
# were scaled, so a later notebook can rebuild predictions on the score scale.
# Both math scores are (T-score - math_center) / math_scale; SES is as supplied.
settings = {
    "model": "stan/regression.stan",
    "n": len(hsls),
    "outcome": bayes.OUTCOME,
    "predictors": bayes.PREDICTORS,
    "math_center": 50,
    "math_scale": 10,
    "priors": bayes.WEAK,
    "chains": 4,
    "iter_warmup": 1000,
    "iter_sampling": 1000,
}
fw.save_model(settings, "hsls_regression_settings")
print(f"Saved {len(hsls_draws):,} draws to {saved.relative_to(fw.project_root())}")
print(json.dumps(settings, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for name, ax in zip(["alpha", "beta[1]", "beta[2]", "sigma"], axes):
    sns.histplot(hsls_draws[name], bins=50, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("")
axes[0].set_ylabel("Draws")
fig.suptitle(f"Marginal posterior draws, HSLS:09 (n = {len(hsls):,}, 4,000 draws)")
fig.tight_layout()

Each histogram above displays the marginal distribution of one parameter across
its 4,000 post-warmup draws, showing the center and spread of plausible values.
Across parameters, aligned iterations form a sample from the joint posterior
distribution, preserving how uncertainty covaries from one parameter to
another. Every summary in the later notebooks, from a parameter mean to a
predicted score for a new student, is computed once per draw and then
summarized across draws. Before any of those summaries are interpreted,
`06-diagnostics` checks whether the draws are trustworthy.